# Getting Started: CS549 Traffic Forecasting

This notebook is a short guided tour of the released data:

1. Load the training data
2. Inspect array dimensions
3. Plot traffic speed for a few sensors
4. Produce a simple validation prediction (naive persistence baseline)
5. Compute validation MAE
6. Create a correctly formatted `submission.csv`

It does **not** use any hidden test labels -- only `train.npz` and
`validation.npz` have labels, and `test_features.npz` (used only in step 6)
has none.

In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from starter_code.load_data import load_split
from starter_code.evaluate_validation import evaluate_predictions
from starter_code.create_submission import build_submission

## 1. Load the training data

In [2]:
train = load_split("train")
list(train.keys())

['X', 'Y', 'valid_mask', 'timestamps', 'sensor_ids', 'sample_id']

## 2. Inspect dimensions

In [3]:
for key, value in train.items():
    print(f"{key:12s} shape={value.shape} dtype={value.dtype}")

n_samples, history_length, num_sensors = train["X"].shape
forecast_horizon = train["Y"].shape[1]
print(f"\n{n_samples} training samples, {history_length} history steps, "
      f"{forecast_horizon} forecast steps, {num_sensors} sensors")

X            shape=(20549, 12, 207) dtype=float32
Y            shape=(20549, 12, 207) dtype=float32
valid_mask   shape=(20549, 207) dtype=bool
timestamps   shape=(20549,) dtype=datetime64[ns]
sensor_ids   shape=(207,) dtype=<U16
sample_id    shape=(20549,) dtype=<U13

20549 training samples, 12 history steps, 12 forecast steps, 207 sensors


## 3. Plot traffic speed for a few sensors

We reconstruct a short stretch of the raw time series for 3 sensors by
stitching together consecutive training windows.

In [4]:
sensor_indices = [0, 1, 2]
n_windows_to_plot = 40  # ~ a few hours, since windows overlap (stride=1)

fig, ax = plt.subplots(figsize=(10, 4))
for s in sensor_indices:
    # X[:, -1, s] is the most recent observed speed at each forecast origin;
    # walking it forward gives a (roughly 5-minute-spaced) speed trace.
    trace = train["X"][:n_windows_to_plot, -1, s]
    ax.plot(trace, label=f"sensor {s} ({train['sensor_ids'][s]})")

ax.set_xlabel("time step (5 min each)")
ax.set_ylabel("speed (mph)")
ax.set_title("Traffic speed, first few training windows")
ax.legend()
plt.show()

## 4. A simple validation prediction

As a first, deliberately naive model: predict that speed stays constant at
its last observed value for all 12 future steps ("persistence").

In [5]:
val = load_split("validation")

def persistence_predict(X, forecast_horizon):
    last_observed = X[:, -1:, :]
    return np.repeat(last_observed, forecast_horizon, axis=1)

val_pred = persistence_predict(val["X"], forecast_horizon)
val_pred.shape

(428, 12, 207)

## 5. Compute validation MAE

`evaluate_predictions` scores only `(sample, sensor)` pairs where
`valid_mask` is True -- see `DATA_DESCRIPTION.md` for why.

In [6]:
metrics = evaluate_predictions(val_pred)
metrics

Scored on 1028556/1063152 valid entries:
    MAE : 3.8694
    RMSE: 7.5340
    MAPE: 9.5766%


{'MAE': 3.8693642407028177,
 'RMSE': 7.533960600752727,
 'MAPE': 9.576603670757708}

## 6. Create a submission.csv

`test_features.npz` has no labels -- we only have `X`. We reuse the same
naive persistence rule and hand the resulting tensor to
`build_submission`, which reads the required Ids straight from
`data/sample_submission.csv` and validates the result.

In [7]:
test = load_split("test_features")
test_pred = persistence_predict(test["X"], forecast_horizon)

submission = build_submission(test_pred, out_path="submission.csv")
submission.head()

Submission is valid: correct row count, correct/ordered Ids, no NaN/Inf values.


Wrote 1517292-row submission to submission.csv


,Id,Prediction
0,sample_000001_sensor_001_h01,63.0
1,sample_000001_sensor_001_h02,63.0
2,sample_000001_sensor_001_h03,63.0
3,sample_000001_sensor_001_h04,63.0
4,sample_000001_sensor_001_h05,63.0


## Next steps

This notebook only demonstrates the naive persistence baseline. For your
project you need at least two baselines, a deep learning model, and an
improved final model -- see `baselines/` for two more worked examples and
`PROJECT_DESCRIPTION.md` for the full assignment.